<a href="https://colab.research.google.com/github/Kevin-March/Tesis/blob/testing/dpo_mistral.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# NB3-a · `dpo_mistral.ipynb` — Generación de candidatos (K=5) + smoke-test

**Fase 2 (DPO / HitL).** Notebook 3 del pipeline (NB1 → NB2 → **NB3** → NB4).

- **Modelo base:** `unsloth/mistral-7b-instruct-v0.3-bnb-4bit` — Camino A: DPO directo sobre el
  instruct, sin SFT propio (§17.11). El instruction-tuning cubre el paso SFT del pipeline RLHF.
- **Insumos:** `contextos.json` (top-2 de `B_graphrag`) + `set_gold_FINAL.py` (las 50 de `PREGUNTAS_BELEN`).
  Se excluyen `CASOS_TESTIGO` (test set, §17.9c) y `PREGUNTAS_FASE2` (sin contexto congelado aún, §17.11).
- **Salida de NB3-a:** `candidatos.json` — 5 candidatos por pregunta, para que los abogados los rankeen.
- **NB3-b (después):** entrenamiento DPO, tras recibir `preferencias.json` (rankings).

**System canónico:** el mismo del agente de NB1 (celda 28), compartido con NB4 vía `system_canonico.txt`.
Se mantiene **neutro en tono** a propósito: la claridad la aprende el modelo por DPO, no por prompt.

---
### Flujo de sesión (importante)
1. Correr **Celda 0** (instalar) → **Entorno de ejecución → Reiniciar sesión**.
2. Correr **Celda 1 → 2 → 3** (setup, insumos, smoke-test).
3. Si el smoke-test pasa: **reiniciar sesión** de nuevo y correr **1 → 2 → 4** para la generación real
   (el smoke deja un modelo con LoRA dummy cargado; conviene arrancar la generación en limpio).

Requisitos: runtime con **GPU T4** y Drive con `contextos.json` + `set_gold_FINAL.py` en `tesis_chatbot`.
No hay cuentas ni API keys que crear: Unsloth/TRL/transformers/peft son librerías `pip`, y el modelo es público.

## Celda 0 — Instalación
Correr **sola**, y después **reiniciar la sesión**. Una vez por sesión de Colab.

In [ ]:
# ============================================================================
# NB3-a · CELDA 0 — Instalación (correr SOLA, luego REINICIAR sesión)
# ============================================================================
!pip install -q unsloth unsloth_zoo
# >>> Al terminar: Entorno de ejecución -> Reiniciar sesión, y seguir en la celda 1.
# El reinicio es necesario para que el PyTorch de Unsloth quede limpio en memoria.
# (El warning de 'cuda-bindings' es cosmético — ignoralo.)

## Celda 1 — Setup, GPU y artefacto canónico
`import unsloth` va **primero de todo**, antes de `torch`/`transformers` (si no, Unsloth no parchea bien y aparece el error de `_c10d_functional`).

In [ ]:
# ============================================================================
# NB3-a · CELDA 1 — Setup, GPU y artefacto canónico compartido
# ============================================================================
import unsloth                      # <-- PRIMERO DE TODO, antes de torch/transformers
import torch
assert torch.cuda.is_available(), "No hay GPU. Entorno de ejecución -> Cambiar tipo -> T4."
print("GPU:", torch.cuda.get_device_name(0),
      "|", round(torch.cuda.get_device_properties(0).total_memory/1e9, 1), "GB")

from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_DIR  = "/content/drive/MyDrive/tesis_chatbot"
CONTEXTOS  = os.path.join(DRIVE_DIR, "contextos.json")
SET_GOLD   = os.path.join(DRIVE_DIR, "set_gold_FINAL.py")
SYS_TXT    = os.path.join(DRIVE_DIR, "system_canonico.txt")
CANDIDATOS = os.path.join(DRIVE_DIR, "candidatos.json")

# Artefacto canónico: system del agente de NB1 (celda 28), palabra por palabra.
# NB4 debe LEER este archivo, no redefinir el system.
SYSTEM_CANONICO = (
    "Sos un asistente legal sobre derecho de inversiones de Paraguay. "
    "Respondé usando SOLO el contexto provisto. Citá la norma y el artículo en cada afirmación. "
    "Si una norma figura como DEROGADA o con estado distinto de vigente, aclaralo y NO la "
    "presentes como vigente. Si una FICHA advierte que referencia normas no vigentes, "
    "trasladá esa advertencia. Si el contexto no alcanza, decilo."
)
with open(SYS_TXT, "w", encoding="utf-8") as f:
    f.write(SYSTEM_CANONICO)
print("system_canonico.txt escrito.")

for p in (CONTEXTOS, SET_GOLD):
    print(("OK    " if os.path.exists(p) else "FALTA ") + p)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


W1111 02:38:06.713000 848 torch/utils/_pytree.py:630] <enum 'KernelPreference'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.
W1111 02:38:06.860000 848 torch/utils/_pytree.py:630] <enum 'ScaleCalculationMode'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.


Unsloth: xformers is installed but cannot load with torch 2.14.1+cu130 (it was built for 2.10.0+cu128), so PyTorch attention is used instead.
To install a matching xformers, run `unsloth install-kernels xformers`, or `pip uninstall xformers` to hide this. UNSLOTH_ENABLE_LOGGING=1 shows the full error.
🦥 Unsloth Zoo will now patch everything to make training faster!
GPU: Tesla T4 | 15.6 GB
Mounted at /content/drive
system_canonico.txt escrito.
OK    /content/drive/MyDrive/tesis_chatbot/contextos.json
OK    /content/drive/MyDrive/tesis_chatbot/set_gold_FINAL.py


## Celda 2 — Insumos: las 50 de BELEN con contexto top-2 de B
Prompt = join por ID: pregunta (`set_gold`) + top-2 de `B_graphrag` (`contextos.json`). Formato de input idéntico al nodo `responder` de NB1.

In [ ]:
# ============================================================================
# NB3-a · CELDA 2 — Las 50 de BELEN con contexto top-2 de B_graphrag
# ============================================================================
import json, importlib.util, statistics as st

with open(CONTEXTOS, encoding="utf-8") as f:
    CTX = json.load(f)["contextos"]

spec = importlib.util.spec_from_file_location("set_gold_FINAL", SET_GOLD)
sg = importlib.util.module_from_spec(spec); spec.loader.exec_module(sg)

TOP_K_DPO = 2   # §17.8: top-2 para el DPO (no 5)

def contexto_top2(qid):
    return "\n\n".join(CTX[qid]["B_graphrag"][:TOP_K_DPO])

def input_canonico(ctx, preg):     # idéntico a NB1 celda 28, línea 75
    return f"# Contexto:\n{ctx}\n\n# Pregunta:\n{preg}"

trabajo = []
for q in sg.PREGUNTAS_BELEN + sg.PREGUNTAS_FASE2:
    qid = q["id"]
    if qid not in CTX:
        print("sin contexto:", qid); continue
    ctx = contexto_top2(qid)
    trabajo.append({"id": qid, "pregunta": q["pregunta"], "contexto": ctx,
                    "input": input_canonico(ctx, q["pregunta"])})

print(f"Preguntas de trabajo: {len(trabajo)}  (esperado 93)")
print("Contextos vacíos:", [t['id'] for t in trabajo if not t['contexto'].strip()] or "ninguno")

largos = sorted(trabajo, key=lambda t: len(t["input"]), reverse=True)
print("input chars -> max:", len(largos[0]["input"]),
      "| mediana:", int(st.median([len(t["input"]) for t in trabajo])))
print("Top-3 más largos:", [(t["id"], len(t["input"])) for t in largos[:3]])

## Celda 3 — Smoke-test de OOM
Puramente **diagnóstico y descartable**: no guarda nada que se use después. Responde una sola
pregunta: **¿la T4 aguanta el DPO con `max_seq_length=3072`?** Verde (< ~13 GB) → seguir. Rojo por
memoria → bajar a 2560 o reducir `r`. (Con 4096 el pico dio 14.1/15 GB — al límite, por eso 3072.)

In [ ]:
# ============================================================================
# NB3-a · CELDA 3 — SMOKE-TEST de OOM
# ============================================================================
from unsloth import FastLanguageModel, PatchDPOTrainer, is_bfloat16_supported
PatchDPOTrainer()
from trl import DPOTrainer, DPOConfig
from datasets import Dataset

MAX_SEQ, MAX_PROMPT = 3072, 2048          # peor prompt real ~1860 tok; deja ~1024 para respuesta

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/mistral-7b-instruct-v0.3-bnb-4bit",
    max_seq_length=MAX_SEQ,
    dtype=None,                            # None -> fp16 en T4 (no soporta bf16)
    load_in_4bit=True,
)
model = FastLanguageModel.get_peft_model(
    model, r=16,
    target_modules=["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"],
    lora_alpha=16, lora_dropout=0, bias="none",
    use_gradient_checkpointing="unsloth", random_state=3407,
)

with open(SYS_TXT, encoding="utf-8") as f:
    SYSTEM_CANONICO = f.read()

def to_prompt(inp):
    msgs = [{"role":"system","content":SYSTEM_CANONICO},
            {"role":"user","content":inp}]
    return tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)

relleno = "Según el contexto, la norma aplicable es la citada y su artículo. " * 60
dummy = [{"prompt": to_prompt(t["input"]), "chosen": relleno, "rejected": relleno[:200]}
         for t in largos[:4]]              # los 4 inputs más largos reales
ds = Dataset.from_list(dummy)

cfg = DPOConfig(
    output_dir="/content/smoke",
    per_device_train_batch_size=1, gradient_accumulation_steps=4,
    max_steps=3, learning_rate=5e-6, beta=0.1,
    max_length=MAX_SEQ, max_prompt_length=MAX_PROMPT,
    fp16=not is_bfloat16_supported(), bf16=is_bfloat16_supported(),
    optim="adamw_8bit", logging_steps=1, report_to="none", seed=42,
)
trainer = DPOTrainer(
    model=model, ref_model=None,           # PEFT+Unsloth: referencia = adaptador off
    args=cfg, train_dataset=ds, processing_class=tokenizer,
)

torch.cuda.reset_peak_memory_stats()
trainer.train()
pico = torch.cuda.max_memory_reserved()/1e9
print(f"\n>>> SMOKE-TEST OK. VRAM pico: {pico:.1f}/15 GB. "
      f"{'Margen cómodo.' if pico < 13 else 'AL LÍMITE: bajar max_seq_length.'}")

## Celda 4 — Generación de K=5 candidatos (PILOTO)
**Recomendado: reiniciar la sesión después del smoke-test** y correr `1 → 2 → 4`. Genera con el
instruct **base (sin LoRA)**. La instrucción de estilo se usa solo para generar diversidad y **no** se
guarda como prompt del DPO (§17.3). Piloto de 3 preguntas (una con derogación de por medio, P1-06)
para inspeccionar el abanico antes de las 50. Estilos con polos exagerados y `max_new_tokens=768`.

In [ ]:
# ============================================================================
# NB3-a · CELDA 4 — Generación de K=5 candidatos por pregunta (PILOTO)
# ============================================================================
import gc, torch
for _v in ("trainer", "model", "tokenizer", "ds"):   # limpieza defensiva
    globals().pop(_v, None)
gc.collect(); torch.cuda.empty_cache()

from unsloth import FastLanguageModel
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/mistral-7b-instruct-v0.3-bnb-4bit",
    max_seq_length=3072, dtype=None, load_in_4bit=True,
)
FastLanguageModel.for_inference(model)          # modo inferencia (2x más rápido)

with open(SYS_TXT, encoding="utf-8") as f:
    SYSTEM_CANONICO = f.read()

# Los 5 estilos (nombre, temperatura, instrucción de estilo)
ESTILOS = [
    {"nombre": "literal",     "temp": 0.0,
     "instr": "Transcribí textualmente los artículos aplicables entre comillas, sin ninguna introducción, explicación ni reformulación. No agregues nada de tu parte."},
    {"nombre": "legalista",   "temp": 0.3,
     "instr": "Respondé con lenguaje jurídico formal y técnico."},
    {"nombre": "explicativa", "temp": 0.7,
     "instr": "Explicá de forma clara y accesible para un inversor sin formación jurídica, sin perder precisión."},
    {"nombre": "incompleta",  "temp": 0.7,
     "instr": "Respondé en una o dos oraciones, sin citar artículos y sin mencionar si alguna norma está derogada."},
    {"nombre": "generica",    "temp": 1.0,
     "instr": "Respondé de forma vaga y general, sin citar ninguna norma ni artículo."},
]

def generar(input_canonico, instr, temp, max_new=1280):
    # La instrucción de estilo va en el turno del USUARIO; el system se mantiene canónico e intacto.
    user = input_canonico + f"\n\n[Estilo de respuesta: {instr}]"
    msgs = [{"role": "system", "content": SYSTEM_CANONICO},
            {"role": "user",   "content": user}]
    prompt = tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
    do_sample = temp > 0
    out = model.generate(
        **inputs, max_new_tokens=max_new,
        do_sample=do_sample,
        temperature=(temp if do_sample else None),
        top_p=(0.9 if do_sample else None),
        pad_token_id=tokenizer.eos_token_id,
    )
    return tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()

# PILOTO: 3 preguntas (P1-06 tiene derogación de por medio: 60/90 DEROGADA por 7548/25)
torch.manual_seed(42)
IDS_PILOTO = ["P1-01", "P1-02", "P1-06"]
PILOTO = [t for t in trabajo if t["id"] in IDS_PILOTO]
for t in PILOTO:
    print("="*90)
    print(f"[{t['id']}] {t['pregunta']}")
    print("-"*90)
    for e in ESTILOS:
        txt = generar(t["input"], e["instr"], e["temp"])
        print(f"\n### {e['nombre'].upper()} (temp={e['temp']})\n{txt}")
    print()

## Celda 5 — Corrida completa (50) + `candidatos.json`  [reanudable]
Genera los 5 estilos para las 50 preguntas (250 generaciones, ~1,5–2 h en T4). **Guarda cada 5 preguntas
y retoma** si la sesión se corta (re-correr `1 → 2 → 4 → 5` y saltea lo ya hecho). Escritura atómica
(no corrompe el JSON si se corta a mitad). Guarda por pregunta el `input` canónico + los 5 candidatos, y
el `system` en `meta` — **sin** la línea de estilo (§17.3). **Correr después de la celda 4.**

In [ ]:
# ============================================================================
# NB3-a · CELDA 5 — Corrida completa (50) + candidatos.json  [REANUDABLE]
# ============================================================================
# 50 x 5 = 250 generaciones (~1.5-2 h en T4). Guarda cada GUARDAR_CADA preguntas
# y retoma desde donde quedó si la sesión se corta. Un error por pregunta no tumba
# la corrida (try/except). El prompt del DPO = system (meta) + input, SIN la línea
# de estilo (§17.3); NB3-b re-arma el canónico. Requiere la celda 4 corrida antes
# (usa model, tokenizer, generar, ESTILOS, SYSTEM_CANONICO).
import json, os, time, torch

GUARDAR_CADA = 5

# Reanudar: cargar lo ya generado (si existe)
if os.path.exists(CANDIDATOS):
    with open(CANDIDATOS, encoding="utf-8") as f:
        salida = json.load(f)
    hechas = set(salida["candidatos"].keys())
    print(f"Reanudando: {len(hechas)} preguntas ya generadas.")
else:
    salida = {
        "meta": {
            "modelo": "unsloth/mistral-7b-instruct-v0.3-bnb-4bit",
            "system_canonico": SYSTEM_CANONICO,
            "estilos": [e["nombre"] for e in ESTILOS],
            "max_new_tokens": 1280,
            "top_k_dpo": TOP_K_DPO,
            "seed": 42,
            "generado": None,
        },
        "candidatos": {},
    }
    hechas = set()

pendientes = [t for t in trabajo if t["id"] not in hechas]
print(f"Pendientes: {len(pendientes)} de {len(trabajo)}")

def guardar():
    salida["meta"]["generado"] = time.strftime("%Y-%m-%dT%H:%M:%S")
    tmp = CANDIDATOS + ".tmp"
    with open(tmp, "w", encoding="utf-8") as f:
        json.dump(salida, f, ensure_ascii=False, indent=2)
    os.replace(tmp, CANDIDATOS)          # escritura atómica: no corrompe si se corta

torch.manual_seed(42)
t0 = time.time()
for i, t in enumerate(pendientes, 1):
    try:
        cands = [{"estilo": e["nombre"], "texto": generar(t["input"], e["instr"], e["temp"])}
                 for e in ESTILOS]
        salida["candidatos"][t["id"]] = {
            "pregunta": t["pregunta"],
            "contexto": t["contexto"],
            "input": t["input"],
            "candidatos": cands,
        }
    except Exception as ex:
        print(f"  ERROR en {t['id']}: {ex}")
        continue
    if i % GUARDAR_CADA == 0:
        guardar()
        print(f"[{i}/{len(pendientes)}] guardado (última: {t['id']}) "
              f"~{(time.time()-t0)/i:.0f}s/preg")

guardar()
n = len(salida["candidatos"])
print(f"\n>>> LISTO. {n}/{len(trabajo)} preguntas con 5 candidatos -> candidatos.json")
if n < len(trabajo):
    faltan = [t['id'] for t in trabajo if t['id'] not in salida['candidatos']]
    print("Faltan (re-correr la celda para retomar):", faltan)

---
# PARTE B — Entrenamiento DPO (NB3-b)

De las preferencias de los abogados al adaptador entrenado. **No depende de la GPU hasta la celda 7.**

**Flujo:** reiniciar sesión limpia → **celda 1** (setup/paths) → **celda 6** (procesa rankings → `preferencias.json`, CPU) → **celda 7** (cargar modelo, GPU) → **celda 8** (entrenar + guardar adaptador) → **celda 9** (prueba base vs DPO). No hacen falta las celdas 2-5 (esas son de generación).

Estructura esperada en Drive:
```
tesis_chatbot/
├── tramos/tramo1_data.json              (fuente de verdad del tramo, NO se comparte)
├── rankings/tramo1_abogadoA.xlsx        (lo devuelve el abogado A)
├── rankings/tramo1_abogadoB.xlsx        (lo devuelve el abogado B)
├── preferencias.json                    (lo genera la celda 6)
└── adaptador_dpo_ciclo1/                (lo genera la celda 8)
```

## Celda 6 — Rankings → `preferencias.json` + Cohen's Kappa (CPU)
Lee el `tramo1_data.json` y los dos xlsx completados, promedia las 6 dimensiones por candidato, deriva el ranking (desempate por Precisión legal), descarta empates y pares con margen < 0,5, y reporta el κ ponderado.

In [ ]:
# ============================================================================
# NB3-b · CELDA 6 — Rankings de abogados -> preferencias.json + Cohen's Kappa
# ============================================================================
import json, itertools
from openpyxl import load_workbook
from sklearn.metrics import cohen_kappa_score

TRAMO_DATA    = f"{DRIVE_DIR}/tramos/tramo1_data.json"
XLSX_ABOGADOS = [f"{DRIVE_DIR}/rankings/tramo1_abogadoA.xlsx",
                 f"{DRIVE_DIR}/rankings/tramo1_abogadoB.xlsx"]
PREFERENCIAS  = f"{DRIVE_DIR}/preferencias.json"
UMBRAL_MARGEN = 0.5          # descarta pares con diferencia de promedio < esto (§17.13)
DIMS = ["Precisión legal","Claridad","Utilidad","Coherencia","Confianza","Trazabilidad"]
ETIQ = ['A','B','C','D','E']

data = json.load(open(TRAMO_DATA, encoding="utf-8"))
orden = [p['id'] for p in data['preguntas']]

def leer_xlsx(path):
    ws = load_workbook(path, data_only=True)['Evaluación']
    filas = []
    for row in ws.iter_rows(values_only=True):
        if row and str(row[0]).strip() in ETIQ:
            nums = row[2:8]
            if all(isinstance(x,(int,float)) for x in nums):
                filas.append((str(row[0]).strip(), [float(x) for x in nums]))
    assert len(filas) == len(orden)*5, f"{path}: {len(filas)} filas (esperaba {len(orden)*5}) — ¿reordenaron o borraron filas?"
    return {qid:{et:sc for et,sc in filas[i*5:(i+1)*5]} for i,qid in enumerate(orden)}

punt = [leer_xlsx(p) for p in XLSX_ABOGADOS]

kappa={}
for di,dim in enumerate(DIMS):
    a=[punt[0][q][e][di] for q in orden for e in ETIQ]
    b=[punt[1][q][e][di] for q in orden for e in ETIQ]
    kappa[dim]=round(cohen_kappa_score(a,b,weights='quadratic'),3)
ga=[punt[0][q][e][di] for q in orden for e in ETIQ for di in range(6)]
gb=[punt[1][q][e][di] for q in orden for e in ETIQ for di in range(6)]
kappa_global=round(cohen_kappa_score(ga,gb,weights='quadratic'),3)

pares=[]; resumen={}
for p in data['preguntas']:
    qid=p['id']
    tex={r['etiqueta']:r['texto'] for r in p['respuestas']}
    est={r['etiqueta']:r['estilo'] for r in p['respuestas']}
    score={}; prec={}
    for e in ETIQ:
        vals=[punt[0][qid][e], punt[1][qid][e]]
        prom6=[(vals[0][i]+vals[1][i])/2 for i in range(6)]
        score[e]=sum(prom6)/6; prec[e]=prom6[0]
    rank=sorted(ETIQ,key=lambda e:(score[e],prec[e]),reverse=True)
    for ch,rej in itertools.combinations(rank,2):
        margen=round(score[ch]-score[rej],3)
        if margen < UMBRAL_MARGEN: continue
        pares.append({"id":qid,"input":p['input'],"chosen":tex[ch],"rejected":tex[rej],
                      "margen":margen,"chosen_estilo":est[ch],"rejected_estilo":est[rej]})
        k=(est[ch],est[rej]); resumen[k]=resumen.get(k,0)+1

pref={"meta":{"ciclo":1,"tramos":["tramo_1"],"n_preguntas":len(orden),"n_pares":len(pares),
              "umbral_margen":UMBRAL_MARGEN,"system_canonico":data['system_canonico']},
      "kappa":{"global":kappa_global,"por_dimension":kappa},"pares":pares}
json.dump(pref, open(PREFERENCIAS,"w",encoding="utf-8"), ensure_ascii=False, indent=2)

print(f"Cohen's Kappa ponderado global: {kappa_global}")
for d,v in kappa.items(): print(f"   {d}: {v}")
print(f"\nPares para el DPO: {len(pares)}  (descartados por empate/margen<{UMBRAL_MARGEN}: {len(orden)*10-len(pares)})")
print("preferencias.json guardado.")

## Celda 7 — Cargar Mistral 4bit + LoRA para entrenar (GPU)
Mismo stack validado en el smoke-test. `r=16`, `max_seq_length=3072`.

In [ ]:
# ============================================================================
# NB3-b · CELDA 7 — Cargar Mistral 4bit + LoRA para ENTRENAR (GPU)
# ============================================================================
import unsloth
from unsloth import FastLanguageModel, PatchDPOTrainer, is_bfloat16_supported
PatchDPOTrainer()
import torch

MAX_SEQ, MAX_PROMPT = 3072, 2048     # techo validado en T4 (§17.12)
R_LORA = 16                          # rank LoRA; subir a 32 solo si falta capacidad

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/mistral-7b-instruct-v0.3-bnb-4bit",
    max_seq_length=MAX_SEQ, dtype=None, load_in_4bit=True,
)
model = FastLanguageModel.get_peft_model(
    model, r=R_LORA,
    target_modules=["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"],
    lora_alpha=R_LORA, lora_dropout=0, bias="none",
    use_gradient_checkpointing="unsloth", random_state=3407,
)
print("Modelo + LoRA listos para entrenar. r =", R_LORA)

## Celda 8 — Dataset DPO + entrenamiento + guardar adaptador (GPU)
Hiperparámetros de arranque conservadores (`beta=0.1`, `lr=5e-6`, `3 epochs`). Mirar `rewards/accuracies` (sube hacia 1) y `rewards/margins` (>0, creciente). Si sobreajusta, bajar epochs o subir `beta`.

In [ ]:
# ============================================================================
# NB3-b · CELDA 8 — Dataset DPO + entrenamiento + guardar adaptador (GPU)
# ============================================================================
import json
from trl import DPOTrainer, DPOConfig
from datasets import Dataset

PREFERENCIAS   = f"{DRIVE_DIR}/preferencias.json"
RUTA_ADAPTADOR = f"{DRIVE_DIR}/adaptador_dpo_ciclo1"
EPOCHS, LR, BETA = 3, 5e-6, 0.1      # arranque; ajustar según rewards/accuracies

pref = json.load(open(PREFERENCIAS, encoding="utf-8"))
SYSTEM_CANONICO = pref["meta"]["system_canonico"]

def to_prompt(inp):
    msgs = [{"role":"system","content":SYSTEM_CANONICO},{"role":"user","content":inp}]
    return tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)

ds = Dataset.from_list([
    {"prompt": to_prompt(p["input"]), "chosen": p["chosen"], "rejected": p["rejected"]}
    for p in pref["pares"]
])
print(f"Pares de entrenamiento: {len(ds)} | kappa global: {pref['kappa']['global']}")

cfg = DPOConfig(
    output_dir="/content/dpo_ckpt",
    per_device_train_batch_size=1, gradient_accumulation_steps=4,
    num_train_epochs=EPOCHS, learning_rate=LR, beta=BETA,
    max_length=MAX_SEQ, max_prompt_length=MAX_PROMPT,
    warmup_ratio=0.1, lr_scheduler_type="cosine",
    fp16=not is_bfloat16_supported(), bf16=is_bfloat16_supported(),
    optim="adamw_8bit", logging_steps=1, save_strategy="no",
    report_to="none", seed=42,
)
trainer = DPOTrainer(model=model, ref_model=None, args=cfg,
                     train_dataset=ds, processing_class=tokenizer)

torch.cuda.reset_peak_memory_stats()
trainer.train()

model.save_pretrained(RUTA_ADAPTADOR)        # solo el adaptador LoRA (liviano)
tokenizer.save_pretrained(RUTA_ADAPTADOR)

ult = [l for l in trainer.state.log_history if 'loss' in l][-1]
print(f"\n>>> DPO-1 LISTO. adaptador -> {RUTA_ADAPTADOR}")
print(f"    loss final: {ult.get('loss')} | VRAM pico: {torch.cuda.max_memory_reserved()/1e9:.1f} GB")
print(f"    rewards/accuracies: {ult.get('rewards/accuracies')}  (sube de ~0.5 hacia 1)")
print(f"    rewards/margins: {ult.get('rewards/margins')}  (debe ser > 0 y creciente)")

## Celda 9 — Prueba rápida: base vs DPO (pregunta NO entrenada)
Compara con y sin el adaptador en la misma pregunta, greedy. Usar una pregunta FUERA del tramo entrenado → mide generalización.

In [ ]:
# ============================================================================
# NB3-b · CELDA 9 — Prueba rápida: base vs DPO (sobre pregunta NO entrenada)
# ============================================================================
import json
FastLanguageModel.for_inference(model)

PREGUNTA_PRUEBA = "N-19"      # cambiá por cualquier id que NO esté en el tramo 1
cand = json.load(open(f"{DRIVE_DIR}/candidatos.json", encoding="utf-8"))["candidatos"]
inp = cand[PREGUNTA_PRUEBA]["input"]

def responder(inp):
    msgs = [{"role":"system","content":SYSTEM_CANONICO},{"role":"user","content":inp}]
    prompt = tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    ins = tokenizer(prompt, return_tensors="pt").to("cuda")
    out = model.generate(**ins, max_new_tokens=512, do_sample=False,
                         pad_token_id=tokenizer.eos_token_id)
    return tokenizer.decode(out[0][ins["input_ids"].shape[1]:], skip_special_tokens=True).strip()

print(f"### PREGUNTA ({PREGUNTA_PRUEBA}): {cand[PREGUNTA_PRUEBA]['pregunta']}\n")
print("="*80 + "\n### CON DPO-1 (adaptador activo)\n" + responder(inp))
with model.disable_adapter():
    print("\n" + "="*80 + "\n### BASE (sin DPO)\n" + responder(inp))